In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "1a5b4bf6-c39f-4243-bd0e-24d597fc265b",
   "metadata": {},
   "outputs": [],
   "source": [
    "import requests\n",
    "import base64\n",
    "import json\n",
    "import time\n",
    "from PIL import Image\n",
    "from io import BytesIO\n",
    "from dotenv import load_dotenv \n",
    "import os\n",
    "\n",
    "# --- 1. Configuration ---\n",
    "load_dotenv()\n",
    "\n",
    "api_key = os.getenv(\"RUNPOD_API_KEY\")\n",
    "if not api_key:\n",
    "    raise ValueError(\"API key not found. Please create a .env file and add RUNPOD_API_KEY=your_key\")\n",
    "\n",
    "endpoint_id = os.getenv(\"RUNPOD_ENDPOINT_ID\", \"a48mrbdsbzg35n\")\n",
    "\n",
    "# --- 2. Define the API URLs ---\n",
    "run_url = f\"https://api.runpod.ai/v2/{endpoint_id}/run\"\n",
    "status_url_template = f\"https://api.runpod.ai/v2/{endpoint_id}/status/\"\n",
    "\n",
    "# --- 3. Define the SD1.5 + LoRA Workflow Payload ---\n",
    "sd15_workflow = {\n",
    "    \"input\": {\n",
    "        \"workflow\": {\n",
    "            \"3\": {\n",
    "                \"inputs\": {\n",
    "                    \"seed\": int(time.time() * 1000) % (2**32),  # Random seed\n",
    "                    \"steps\": 20,\n",
    "                    \"cfg\": 8,\n",
    "                    \"sampler_name\": \"euler\",\n",
    "                    \"scheduler\": \"normal\",\n",
    "                    \"denoise\": 1,\n",
    "                    \"model\": [\"10\", 0],\n",
    "                    \"positive\": [\"6\", 0],\n",
    "                    \"negative\": [\"7\", 0],\n",
    "                    \"latent_image\": [\"5\", 0]\n",
    "                },\n",
    "                \"class_type\": \"KSampler\",\n",
    "                \"_meta\": {\n",
    "                    \"title\": \"KSampler\"\n",
    "                }\n",
    "            },\n",
    "            \"4\": {\n",
    "                \"inputs\": {\n",
    "                    \"ckpt_name\": \"v1-5-pruned-emaonly.ckpt\"\n",
    "                },\n",
    "                \"class_type\": \"CheckpointLoaderSimple\",\n",
    "                \"_meta\": {\n",
    "                    \"title\": \"Load Checkpoint\"\n",
    "                }\n",
    "            },\n",
    "            \"5\": {\n",
    "                \"inputs\": {\n",
    "                    \"width\": 512,\n",
    "                    \"height\": 512,\n",
    "                    \"batch_size\": 1\n",
    "                },\n",
    "                \"class_type\": \"EmptyLatentImage\",\n",
    "                \"_meta\": {\n",
    "                    \"title\": \"Empty Latent Image\"\n",
    "                }\n",
    "            },\n",
    "            \"6\": {\n",
    "                \"inputs\": {\n",
    "                    \"text\": \"masterpiece best quality girl\",\n",
    "                    \"clip\": [\"10\", 1]\n",
    "                },\n",
    "                \"class_type\": \"CLIPTextEncode\",\n",
    "                \"_meta\": {\n",
    "                    \"title\": \"CLIP Text Encode (Prompt)\"\n",
    "                }\n",
    "            },\n",
    "            \"7\": {\n",
    "                \"inputs\": {\n",
    "                    \"text\": \"bad hands\",\n",
    "                    \"clip\": [\"10\", 1]\n",
    "                },\n",
    "                \"class_type\": \"CLIPTextEncode\",\n",
    "                \"_meta\": {\n",
    "                    \"title\": \"CLIP Text Encode (Prompt)\"\n",
    "                }\n",
    "            },\n",
    "            \"8\": {\n",
    "                \"inputs\": {\n",
    "                    \"samples\": [\"3\", 0],\n",
    "                    \"vae\": [\"4\", 2]\n",
    "                },\n",
    "                \"class_type\": \"VAEDecode\",\n",
    "                \"_meta\": {\n",
    "                    \"title\": \"VAE Decode\"\n",
    "                }\n",
    "            },\n",
    "            \"9\": {\n",
    "                \"inputs\": {\n",
    "                    \"filename_prefix\": \"ComfyUI_SD15\",\n",
    "                    \"images\": [\"8\", 0]\n",
    "                },\n",
    "                \"class_type\": \"SaveImage\",\n",
    "                \"_meta\": {\n",
    "                    \"title\": \"Save Image\"\n",
    "                }\n",
    "            },\n",
    "            \"10\": {\n",
    "                \"inputs\": {\n",
    "                    \"lora_name\": \"epiNoiseoffset_v2.safetensors\",\n",
    "                    \"strength_model\": 1,\n",
    "                    \"strength_clip\": 1,\n",
    "                    \"model\": [\"4\", 0],\n",
    "                    \"clip\": [\"4\", 1]\n",
    "                },\n",
    "                \"class_type\": \"LoraLoader\",\n",
    "                \"_meta\": {\n",
    "                    \"title\": \"Load LoRA\"\n",
    "                }\n",
    "            }\n",
    "        }\n",
    "    }\n",
    "}\n",
    "\n",
    "# --- 4. Send the Initial API Request to Start the Job ---\n",
    "headers = {\n",
    "    \"Authorization\": f\"Bearer {api_key}\",\n",
    "    \"Content-Type\": \"application/json\"\n",
    "}\n",
    "\n",
    "print(\"Sending request to start the SD1.5 + LoRA job...\")\n",
    "response = requests.post(run_url, headers=headers, json=sd15_workflow)\n",
    "\n",
    "if response.status_code == 200:\n",
    "    response_data = response.json()\n",
    "    job_id = response_data.get('id')\n",
    "    \n",
    "    if not job_id:\n",
    "        print(\"Error: API response did not include a job ID.\")\n",
    "        print(\"Full response:\", json.dumps(response_data, indent=2))\n",
    "    else:\n",
    "        print(f\"Job successfully started with ID: {job_id}\")\n",
    "\n",
    "        # --- 5. Poll for the Job Status ---\n",
    "        status_url = status_url_template + job_id\n",
    "        poll_count = 0\n",
    "        max_polls = 120  # 10 minutes max (5 second intervals)\n",
    "        \n",
    "        while poll_count < max_polls:\n",
    "            print(f\"Polling for job status... (attempt {poll_count + 1})\")\n",
    "            status_response = requests.get(status_url, headers=headers)\n",
    "            status_data = status_response.json()\n",
    "            job_status = status_data.get('status')\n",
    "\n",
    "            if job_status == 'COMPLETED':\n",
    "                print(\"Job completed successfully!\")\n",
    "                \n",
    "                # --- 6. Process the Final Output ---\n",
    "                try:\n",
    "                    output_image_data = status_data['output']['images'][0]\n",
    "                    image_base64 = output_image_data['data'] \n",
    "                    filename = output_image_data['filename']\n",
    "\n",
    "                    image_bytes = base64.b64decode(image_base64)\n",
    "                    image = Image.open(BytesIO(image_bytes))\n",
    "                    \n",
    "                    image.save(filename)\n",
    "                    print(f\"\\nImage successfully generated and saved as '{filename}'\\n\")\n",
    "                    \n",
    "                    # Display the image in the notebook\n",
    "                    from IPython.display import display\n",
    "                    display(image)\n",
    "\n",
    "                except (KeyError, IndexError, TypeError) as e:\n",
    "                    print(f\"\\nError: Could not parse image data from the final response: {e}\\n\")\n",
    "                    \n",
    "                    debug_data = json.loads(json.dumps(status_data))\n",
    "                    try:\n",
    "                        for img in debug_data['output']['images']:\n",
    "                            if 'data' in img:\n",
    "                                original_length = len(img['data'])\n",
    "                                img['data'] = f\"<base64 data redacted, original length: {original_length}>\"\n",
    "                    except Exception:\n",
    "                        pass \n",
    "                        \n",
    "                    print(\"Full response (redacted):\", json.dumps(debug_data, indent=2))\n",
    "                \n",
    "                break\n",
    "\n",
    "            elif job_status in ['IN_QUEUE', 'IN_PROGRESS']:\n",
    "                time.sleep(5)\n",
    "                poll_count += 1\n",
    "            else:\n",
    "                print(f\"Job execution failed or was cancelled.\")\n",
    "                print(\"Final Status:\", job_status)\n",
    "                if 'error' in status_data:\n",
    "                    print(\"Error details:\", status_data['error'])\n",
    "                else:\n",
    "                    print(\"Full response:\", json.dumps(status_data, indent=2))\n",
    "                break \n",
    "        \n",
    "        if poll_count >= max_polls:\n",
    "            print(\"Timeout: Job did not complete within the expected time.\")\n",
    "\n",
    "else:\n",
    "    print(f\"Error: Initial API request failed with status code {response.status_code}\")\n",
    "    print(\"Response:\", response.text)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "550aae1b-b6ef-4869-a8c9-dffc6f8c766c",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}